In [24]:
# imports

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

In [5]:
main_df = pd.read_feather("./Data/US_datastream/US_data_panel_filtered_0.15.feather")
main_df = main_df.sort_values(by="Date")

In [40]:
# metrics for stock_selection 

# for now I also want start with just the pseudo Sharpe
def metric_sharpe(ret_matrix: pd.DataFrame):
    return ret_matrix.mean() / ret_matrix.std(ddof=1) # IMO this is a pretty crazy line since it does so much

In [ ]:
# helpful functions

# function for quickly checking what certain vars actually are at the moment

def get_info(**kwargs):

    for key,value in kwargs.items():
        
        print(key,"|",value, "|", type(value))


# function for changing the format of the initial DataFrame

def construct_ret_matrix(panel: pd.DataFrame) -> pd.DataFrame:

    return panel.pivot(index="Date", columns="DSCD", values="Return")

# filter for stocks with available return data

def filter_av_returns(panel: pd.DataFrame) -> list:

    total_days = panel["Date"].nunique()

    valid_stocks = []
    for dscd, dscd_panel in panel.groupby("DSCD", sort=False):
        
        dscd_filtered_series = dscd_panel["Return"].dropna()
        #dscd_filtered_series = dscd_filtered_series[dscd_filtered_series != 0]

        if len(dscd_filtered_series) != total_days: # I don't allow even a single missing return (but 0 returns are in for now) -> smaler scope helps me here
            continue
        
        valid_stocks.append(dscd)
    
    return valid_stocks

# my individual stock selection function, should also work variable by giving it different metrics

def select_stocks(ret_matrix: pd.DataFrame, nstocks: int, metric: callable, by_highest: bool) -> list:
    
    scores = metric(ret_matrix)
    return scores.sort_values(ascending=(not by_highest)).head(nstocks).index.tolist()

In [ ]:
# TODO dive into detail here, for now just make it work
# TODO e.g. the decision to fill missing returns with 0 is interesting
# TODO e.g. the scaling is interesting 

def sample_cov(R: pd.DataFrame):
    
    Sigma = R.cov().values
    Sigma = Sigma + 1e-4 * np.eye(Sigma.shape[0]) # added this to open quetions, I know what it does but not fully why we do it
    return pd.DataFrame(Sigma, index=R.columns, columns=R.columns)

`sample_cov()`

for stocks i,j compute:

$$\Sigma_{i,j} = \frac{1}{T-1} \sum_t{(r_{i,t} - \bar{r}_i)(r_{j,t} - \bar{r}_j)}$$

In [ ]:
def min_var_weights(Sigma: pd.DataFrame):
    pass

`min_var_weights()`

solve:
$$\min_x{x^T\sum{x}}$$
w.r.t.:
$$\text{no overinvesting}$$
$$\sum_i{x_i}=1$$
$$\text{long-only}$$
$$0 \leq x_i \leq 1$$

In [ ]:
# everything regardings the simulated investors

# my individual investor class

class Investor:

    def __init__(self, strategy: str, cov_estimator: str):
        
        # attributes

        self.strategy = strategy
        self.cov_estimator = cov_estimator
        self.weight_history = []
        self.return_history = []

        if self.strategy == "TODO":
            pass
        else: # default strategy = min_var
            self.weights = min_var_weights
        if self.cov_estimator == "TODO":
            pass
        else: # default covariance estimation is in-sample 
            self.covar = sample_cov

In [47]:
df = main_df[["Date", "Return","DSCD"]]

un_dates = np.sort(df["Date"].unique())
T = len(un_dates)
un_dates_cut = un_dates[2000:6000] # just to scale down a bit for this experimental scope
T_cut = len(un_dates_cut)

estimation_window = 512
holding_period = 21

times_to_rebalance = range(estimation_window, T, holding_period)

for t in tqdm(times_to_rebalance):

    est_start = un_dates_cut[t - estimation_window]
    est_end = un_dates_cut[t - 1]

    prevent_overflow_idx = min(t + holding_period, T_cut)

    holding_period = un_dates_cut[t:prevent_overflow_idx]

    estimation_sector = df[(df["Date"] >= est_start) & (df["Date"] <= est_end)]

    filtered_stocks = filter_av_returns(estimation_sector)
    filtered_sector = estimation_sector[estimation_sector["DSCD"].isin(filtered_stocks)]
    filtered_sector_R = construct_ret_matrix(panel=filtered_sector)

    selected_stocks = select_stocks(ret_matrix=filtered_sector_R, nstocks=10, metric=metric_sharpe, by_highest=True)
    selected_sector = filtered_sector[filtered_sector["DSCD"].isin(selected_stocks)]
    selected_sector_R = filtered_sector_R.loc[:, selected_stocks] # should be the same as just caling construct_ret_matrix again

    selected_sector_R = selected_sector_R.loc[:, selected_sector_R.columns.sort_values()] # not sure why we do this exavtly, I think because rows should match for our matrix calculations later -> added it to open questions

    print(selected_sector_R)

    break

  0%|          | 0/363 [00:02<?, ?it/s]

DSCD          271091    271662    328302    357322    674645    877862  \
Date                                                                     
2001-09-05 -0.005780  0.011555 -0.003858 -0.001556  0.039756 -0.011335   
2001-09-06 -0.011628 -0.000439  0.000775  0.001559 -0.014706 -0.004917   
2001-09-07 -0.047059 -0.030770 -0.000774 -0.002887 -0.023284 -0.001459   
2001-09-10  0.037037 -0.009070  0.006971  0.015383  0.002445 -0.025641   
2001-09-12  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
...              ...       ...       ...       ...       ...       ...   
2003-09-15 -0.038118 -0.003031 -0.001748  0.070990  0.001984 -0.001231   
2003-09-16  0.071918  0.010807  0.003503 -0.036329 -0.001980 -0.005838   
2003-09-17 -0.029667  0.001336  0.063118  0.020503  0.011905  0.019180   
2003-09-18  0.017663  0.012346  0.067032  0.053144 -0.003922  0.023451   
2003-09-19 -0.019667 -0.002307 -0.000257  0.033538  0.000000 -0.009052   

DSCD          895149    921933    930

In [38]:
estimation_sector_R.index

DatetimeIndex(['2001-09-05', '2001-09-06', '2001-09-07', '2001-09-10',
               '2001-09-12', '2001-09-17', '2001-09-18', '2001-09-19',
               '2001-09-20', '2001-09-21',
               ...
               '2003-09-08', '2003-09-09', '2003-09-10', '2003-09-11',
               '2003-09-12', '2003-09-15', '2003-09-16', '2003-09-17',
               '2003-09-18', '2003-09-19'],
              dtype='datetime64[ns]', name='Date', length=512, freq=None)